# Tour API와 LLM(OpenAI)을 활용한 여행 일정 분석 및 추천 테스트 (MVP)

본 노트북은 로컬 정제 데이터(`관광정보_메인_장소_데이터.csv`)에서 실제 장소를 로드하고, 해당 장소의 세부 공통/소개 API 정보를 동적으로 결합하여 OpenAI LLM에 일정 검증 프롬프트를 송신하는 MVP 프로토타입 검증용 노트북입니다.

In [ ]:
!pip install requests pandas python-dotenv openai

In [ ]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

# .env 파일 로드 (상위 폴더 확인)
load_dotenv(dotenv_path="../../.env")
API_KEY = os.getenv("TOUR_API_DECODE_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
BASE_URL = "https://apis.data.go.kr/B551011/KorService2"

print("Tour API 키 상태:", "[SUCCESS] 로드 성공" if API_KEY else "[ERROR] 로드 실패")
print("OpenAI API 키 상태:", "[SUCCESS] 로드 성공" if OPENAI_API_KEY else "[WARNING] 로드 실패 (OpenAI 키가 없는 경우 모의 응답으로 대체 구동됩니다)")

In [ ]:
# 1. 로컬 정제 완료된 관광지 기본 장소 데이터 CSV 로드
csv_path = "../data/관광정보_메인_장소_데이터.csv"
if os.path.exists(csv_path):
    df_places = pd.read_csv(csv_path)
    print(f"[SUCCESS] 로컬 마스터 데이터 로드 완료! (총 {len(df_places)}행)")
    # 예제 검색 테스트: 해운대 관련 명소 1건 추출
    test_target = df_places[df_places['title'].str.contains('해운대', na=False)].head(1)
    display(test_target)
else:
    print("[ERROR] [오류] 관광지 CSV 데이터 파일이 존재하지 않습니다. 먼저 '관광정보_메인_EDA_Pretraining.ipynb'를 실행하세요.")

In [ ]:
# 2. 캐시를 활용한 상세 정보 조회 도구 정의
CACHE_DIR = "../data"
os.makedirs(CACHE_DIR, exist_ok=True)
DETAIL_CACHE_PATH = os.path.join(CACHE_DIR, "detail_intro_cache.json")

def load_detail_cache():
    if os.path.exists(DETAIL_CACHE_PATH):
        try:
            with open(DETAIL_CACHE_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except: pass
    return {}

def save_detail_cache(cache):
    try:
        with open(DETAIL_CACHE_PATH, "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False, indent=2)
    except: pass

def get_detail_common(content_id):
    url = f"{BASE_URL}/detailCommon2"
    params = {
        "serviceKey": API_KEY,
        "MobileOS": "ETC",
        "MobileApp": "RouteCheck",
        "_type": "json",
        "contentId": content_id,
        "defaultYN": "Y",
        "overviewYN": "Y"
    }
    try:
        res = requests.get(url, params=params, timeout=5).json()
        items = res['response']['body']['items']['item']
        return items[0] if isinstance(items, list) else items
    except:
        return {}

def get_detail_intro(content_id, content_type_id):
    cache = load_detail_cache()
    cid_str = str(content_id)
    if cid_str in cache:
        return cache[cid_str]
        
    url = f"{BASE_URL}/detailIntro2"
    params = {
        "serviceKey": API_KEY,
        "MobileOS": "ETC",
        "MobileApp": "RouteCheck",
        "_type": "json",
        "contentId": content_id,
        "contentTypeId": content_type_id
    }
    try:
        res = requests.get(url, params=params, timeout=5).json()
        items = res['response']['body']['items']['item']
        intro = items[0] if isinstance(items, list) else items
        cache[cid_str] = intro
        save_detail_cache(cache)
        return intro
    except:
        return {}

In [ ]:
# 3. 실제 마스터 데이터 매핑 및 프롬프트 생성 테스트
if 'df_places' in locals() and not df_places.empty:
    # 예시: 해운대해수욕장 정보 찾기
    target_place = df_places[df_places['title'].str.contains('해운대', na=False)].head(1)
    if target_place.empty:
        target_place = df_places.head(1)
        
    content_id = int(target_place['contentid'].iloc[0])
    title = target_place['title'].iloc[0]
    content_type_name = target_place['contenttypename'].iloc[0]
    
    type_name_to_id = {
        '관광지': '12', '문화시설': '14', '축제/공연/행사': '15', '여행 코스': '25',
        '레포츠': '28', '숙박': '32', '쇼핑': '38', '음식점': '39'
    }
    content_type_id = type_name_to_id.get(content_type_name, '12')
    
    print(f"🎯 [매핑 대상 장소]: {title} (ID: {content_id}, 타입: {content_type_name})")
    
    # 상세 정보 동적 호출
    detail_common = get_detail_common(content_id)
    detail_intro = get_detail_intro(content_id, content_type_id)
    
    # 운영 및 휴무 텍스트 추출
    usetime = detail_intro.get('usetime', '정보 없음')
    restdate = detail_intro.get('restdate', '정보 없음')
    overview = detail_common.get('overview', '장소 개요 설명이 존재하지 않습니다.')
    
    user_plan = f"내일 월요일 오후 1시에 반려견을 동반하여 {title}에 가서 약 2시간 산책을 즐기려고 해. 이 계획에 문제나 제약조건이 있어?"
    
    # 검증용 LLM 프롬프트 빌드
    prompt = f"""
당신의 역할은 사용자의 여행 계획을 진단하고 조언하는 전문 AI 여행 플래너입니다.
제공된 [관광지 Context]를 기반으로, 사용자의 [여행 계획]의 문제점이나 제약조건을 분석하여 간략한 진단 보고서를 피드백해주세요.

[사용자의 여행 계획]
{user_plan}

[관광지 Context]
- 장소명: {title}
- 운영 및 이용시간: {usetime}
- 쉬는 날 및 휴무일: {restdate}
- 장소 개요: {overview[:150]}...

[출력 요구사항]
1. 진단 요약 (시간대 위반 여부, 요일 휴무 여부, 반려동물 적합성 분석)
2. 최종 계획 점수 (0 ~ 100점)
3. 동선 또는 세부 조정 추천안
"""
    print("\n--- 📝 생성된 LLM Prompt ---")
    print(prompt)
else:
    print("[ERROR] 마스터 데이터가 로드되지 않아 빌드를 스킵합니다.")

In [ ]:
# 4. OpenAI 호출 또는 모의(Mock) 응답 출력 테스트
if 'prompt' in locals():
    if OPENAI_API_KEY:
        from openai import OpenAI
        print("🤖 OpenAI API 호출 시작 (gpt-4o-mini)...")
        client = OpenAI(api_key=OPENAI_API_KEY)
        try:
            res = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2
            )
            print("\n======================= 📝 AI 분석 리포트 =======================\n")
            print(res.choices[0].message.content)
        except Exception as e:
            print("[ERROR] OpenAI 호출 실패:", e)
    else:
        print("[WARNING] OpenAI API 키가 로드되지 않았습니다. 결과 확인용 시뮬레이션 모의 응답을 출력합니다.")
        mock_res = f"""
======================= 📝 AI 분석 리포트 (시뮬레이션 모의 응답) =======================

1. 진단 요약
   - **시간대 위반 여부:** 해수욕장 구역은 상시 개방되어 있으므로 오후 1시 방문은 이용 시간 범위를 준수합니다. (안전)
   - **요일 휴무 여부:** 연중무휴인 관광지로 월요일 방문에 따른 정기 휴일 리스크가 없습니다. (안전)
   - **반려동물 적합성:** {title} 구역은 넓은 개방 공간이지만 백사장 내 반려견 동반 출입은 계절에 따라 금지되거나 규제(목줄, 입마개 의무 및 배변 수거)가 엄격할 수 있으므로 확인이 필요합니다.

2. 최종 계획 점수
   - **80점 / 100점**

3. 동선 또는 세부 조정 추천안
   - 산책 후 반려견 동반 입장이 공식 허용된 인접 카페(부산커피 등)를 경유하는 동선으로 우회 변경하면 안정도가 95점까지 상승합니다.
"""
        print(mock_res)